<a href="https://colab.research.google.com/github/Pes2ug23am092/AIS_transit_procurement_data_analysis/blob/main/Encephalon_AISprocessing_first.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Run this cell if you haven't already installed these in your current Colab session
!pip install geopandas fiona shapely
from shapely.geometry import shape # Required for processing feature geometry
import fiona # Explicitly importing fiona

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.6/56.6 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 83.5 MB/s eta 0:00:00


In [ ]:
#---------------------------------------------------------------------------------------------
# 2021 data load
#---------------------------------------------------------------------------------------------


import geopandas as gpd
import pandas as pd
import time
import os
import fiona
from shapely.geometry import shape

# --- Configuration ---
PATH_2021 = '/content/drive/MyDrive/SupplyChainPairINT/extracted/AISVesselTracks2021.gpkg'
# Using a unique folder for the fixed attempt
OUTPUT_PARQUET_DIR = '/content/drive/MyDrive/SupplyChainPairINT/processed'
CHUNK_SIZE = 1000000

# ***************************************************************
# *** FIX: ASSUMING 'TrackStartTime' is the correct time column
# ***************************************************************
COLUMNS_TO_LOAD = ['MMSI', 'TrackStartTime', 'Draft', 'VesselType', 'geometry']

# LA/Long Beach BBOX (lon_min, lat_min, lon_max, lat_max)
BBOX_LA_LB = (-119.0, 33.0, -117.0, 34.0)

# Ensure output directory exists
os.makedirs(OUTPUT_PARQUET_DIR, exist_ok=True)
print(f"Output directory created: {OUTPUT_PARQUET_DIR}")

# --- 2. Chunked Reading, Filtering, and Saving ---
print(f"Starting chunked processing with FIX. Expect the first chunk to take the longest...")
start_total = time.time()
chunk_index = 0
total_features_processed = 0

try:
    with fiona.open(PATH_2021, 'r', layer=0) as source:

        output_crs = source.crs
        current_data = []

        # Iterate over features filtered by the BBOX
        for feature in source.filter(bbox=BBOX_LA_LB):

            # Use the corrected column names for properties
            properties = {k: feature['properties'][k] for k in COLUMNS_TO_LOAD if k != 'geometry'}

            current_data.append({
                'geometry': shape(feature['geometry']),
                **properties
            })

            total_features_processed += 1

            # Process and save chunk
            if len(current_data) >= CHUNK_SIZE:
                chunk_index += 1
                start_chunk = time.time()

                gdf_chunk = gpd.GeoDataFrame(current_data, geometry='geometry', crs=output_crs)
                output_file = os.path.join(OUTPUT_PARQUET_DIR, f'2021_chunk_{chunk_index:03d}.parquet')
                gdf_chunk.to_parquet(output_file, index=False)

                print(f"   Saved Chunk {chunk_index} ({len(gdf_chunk):,} features) in {time.time() - start_chunk:.2f}s")

                current_data = []

    # Process the last, partial chunk
    if current_data:
        chunk_index += 1
        start_chunk = time.time()
        gdf_chunk = gpd.GeoDataFrame(current_data, geometry='geometry', crs=output_crs)
        output_file = os.path.join(OUTPUT_PARQUET_DIR, f'2021_chunk_{chunk_index:03d}.parquet')
        gdf_chunk.to_parquet(output_file, index=False)
        print(f"   Saved Final Chunk {chunk_index} ({len(gdf_chunk):,} features) in {time.time() - start_chunk:.2f}s")

    total_time = time.time() - start_total
    print(f"\n--- Processing Complete ---")
    print(f"Total time taken: {total_time:.2f} seconds")
    print(f"Total features filtered and processed: {total_features_processed:,}")
    print(f"All relevant 2021 data (LA/LB area) is now saved to {OUTPUT_PARQUET_DIR}")

except Exception as e:
    print(f"An error occurred during processing: {e}")
    # The expected error here, if the fix failed, would be a new column name error.
    print("If this still fails, you must switch to a local machine to proceed.")

Output directory created: /content/drive/MyDrive/SupplyChainPairINT/processed
Starting chunked processing with FIX. Expect the first chunk to take the longest...


   Saved Final Chunk 1 (242,943 features) in 9.39s

--- Processing Complete ---
Total time taken: 106.74 seconds
Total features filtered and processed: 242,943
All relevant 2021 data (LA/LB area) is now saved to /content/drive/MyDrive/SupplyChainPairINT/processed


In [ ]:
#---------------------------------------------------------------------------------------------
# 2021 view
#---------------------------------------------------------------------------------------------


import geopandas as gpd
import pandas as pd
import os

# --- Configuration ---
# Path to the directory where your 2021 data was saved
PARQUET_DIR = '/content/drive/MyDrive/SupplyChainPairINT/processed'

# The filename will be '2021_chunk_001.parquet'
parquet_file = os.path.join(PARQUET_DIR, '2021_chunk_001.parquet')

# FIX: Add 'geometry' to the list of columns to satisfy GeoPandas
COLUMNS_TO_VIEW = ['MMSI', 'TrackStartTime', 'Draft', 'VesselType', 'geometry']

if os.path.exists(parquet_file):
    print(f"Loading sample from: {parquet_file}")

    # Use geopandas to read the data, including the geometry column
    sample_data = gpd.read_parquet(
        parquet_file,
        columns=COLUMNS_TO_VIEW
    ).head(10) # Read only the first 10 rows

    print("\n--- Sample Data from 2021 LA/LB Parquet (First 10 Rows) ---")

    # Display the data. We'll show the DataFrame without the complex geometry for clarity,
    # but it confirms the column exists.
    print(sample_data.drop(columns=['geometry']).to_string())

    print(f"\nData types:\n{sample_data.dtypes}")
else:
    print(f"Error: Parquet file not found at {parquet_file}. Check the folder contents.")

Loading sample from: /content/drive/MyDrive/SupplyChainPairINT/processed/2021_chunk_001.parquet

--- Sample Data from 2021 LA/LB Parquet (First 10 Rows) ---
        MMSI             TrackStartTime  Draft  VesselType
0  636016093  2021-03-19T14:54:59+00:00   15.6        80.0
1  248160000  2021-04-24T04:22:52+00:00   10.5         0.0
2  357051000  2021-01-17T05:44:57+00:00   16.0        70.0
3  636017753  2021-01-17T06:20:14+00:00   14.5        70.0
4  477652300  2021-04-20T00:59:10+00:00   12.5        71.0
5  566828000  2021-04-02T10:53:39+00:00   12.8        70.0
6  219005000  2021-01-18T03:53:29+00:00   13.2        80.0
7  373228000  2021-03-14T17:49:21+00:00    9.0        70.0
8  311015100  2021-02-14T21:43:45+00:00    9.8        70.0
9  311004900  2021-03-21T22:19:22+00:00    9.8        70.0

Data types:
MMSI                 int64
TrackStartTime      object
Draft              float64
VesselType         float64
geometry          geometry
dtype: object


In [ ]:
#---------------------------------------------------------------------------------------------
# 2022 data load
#---------------------------------------------------------------------------------------------


import geopandas as gpd
import pandas as pd
import time
import os
import fiona
from shapely.geometry import shape

# --- Configuration for 2022 ---
PATH_2022 = '/content/drive/MyDrive/SupplyChainPairINT/extracted/AISVesselTracks2022.gpkg'
OUTPUT_PARQUET_DIR_2022 = '/content/drive/MyDrive/SupplyChainPairINT/processed'
CHUNK_SIZE = 1000000
COLUMNS_TO_LOAD = ['MMSI', 'TrackStartTime', 'Draft', 'VesselType', 'geometry']
BBOX_LA_LB = (-119.0, 33.0, -117.0, 34.0)

os.makedirs(OUTPUT_PARQUET_DIR_2022, exist_ok=True)
print(f"Output directory created: {OUTPUT_PARQUET_DIR_2022}")

# --- Chunked Reading, Filtering, and Saving for 2022 ---
print(f"Starting chunked processing for 2022...")
start_total = time.time()
chunk_index = 0
total_features_processed = 0

try:
    with fiona.open(PATH_2022, 'r', layer=0) as source:

        output_crs = source.crs
        current_data = []

        for feature in source.filter(bbox=BBOX_LA_LB):
            properties = {k: feature['properties'][k] for k in COLUMNS_TO_LOAD if k != 'geometry'}

            current_data.append({
                'geometry': shape(feature['geometry']),
                **properties
            })

            total_features_processed += 1

            if len(current_data) >= CHUNK_SIZE:
                chunk_index += 1
                start_chunk = time.time()

                gdf_chunk = gpd.GeoDataFrame(current_data, geometry='geometry', crs=output_crs)
                output_file = os.path.join(OUTPUT_PARQUET_DIR_2022, f'2022_chunk_{chunk_index:03d}.parquet')
                gdf_chunk.to_parquet(output_file, index=False)

                print(f"   Saved Chunk {chunk_index} ({len(gdf_chunk):,} features) in {time.time() - start_chunk:.2f}s")
                current_data = []

    if current_data:
        chunk_index += 1
        start_chunk = time.time()
        gdf_chunk = gpd.GeoDataFrame(current_data, geometry='geometry', crs=output_crs)
        output_file = os.path.join(OUTPUT_PARQUET_DIR_2022, f'2022_chunk_{chunk_index:03d}.parquet')
        gdf_chunk.to_parquet(output_file, index=False)
        print(f"   Saved Final Chunk {chunk_index} ({len(gdf_chunk):,} features) in {time.time() - start_chunk:.2f}s")

    total_time = time.time() - start_total
    print(f"\n--- 2022 Processing Complete ---")
    print(f"Total time taken: {total_time:.2f} seconds")
    print(f"Total features filtered and processed: {total_features_processed:,}")

except Exception as e:
    print(f"An error occurred during 2022 processing: {e}")

Output directory created: /content/drive/MyDrive/SupplyChainPairINT/processed
Starting chunked processing for 2022...


   Saved Final Chunk 1 (269,514 features) in 4.68s

--- 2022 Processing Complete ---
Total time taken: 332.46 seconds
Total features filtered and processed: 269,514


In [ ]:
#---------------------------------------------------------------------------------------------
# 2022 data view
#---------------------------------------------------------------------------------------------

import geopandas as gpd
import pandas as pd
import os

# --- Configuration ---
# Path to the directory where your 2021 data was saved
PARQUET_DIR = '/content/drive/MyDrive/SupplyChainPairINT/processed'

# The filename will be '2021_chunk_001.parquet'
parquet_file = os.path.join(PARQUET_DIR, '2022_chunk_001.parquet')

# FIX: Add 'geometry' to the list of columns to satisfy GeoPandas
COLUMNS_TO_VIEW = ['MMSI', 'TrackStartTime', 'Draft', 'VesselType', 'geometry']

if os.path.exists(parquet_file):
    print(f"Loading sample from: {parquet_file}")

    # Use geopandas to read the data, including the geometry column
    sample_data = gpd.read_parquet(
        parquet_file,
        columns=COLUMNS_TO_VIEW
    ).head(10) # Read only the first 10 rows

    print("\n--- Sample Data from 2021 LA/LB Parquet (First 10 Rows) ---")

    # Display the data. We'll show the DataFrame without the complex geometry for clarity,
    # but it confirms the column exists.
    print(sample_data.drop(columns=['geometry']).to_string())

    print(f"\nData types:\n{sample_data.dtypes}")
else:
    print(f"Error: Parquet file not found at {parquet_file}. Check the folder contents.")

Loading sample from: /content/drive/MyDrive/SupplyChainPairINT/processed/2022_chunk_001.parquet

--- Sample Data from 2021 LA/LB Parquet (First 10 Rows) ---
        MMSI             TrackStartTime  Draft  VesselType
0  338383211  2022-01-23T22:00:09+00:00    NaN        37.0
1  366857070  2022-01-31T23:00:08+00:00    NaN        60.0
2  338383211  2022-01-29T00:00:02+00:00    NaN        37.0
3  366857070  2022-01-18T00:00:37+00:00    NaN        60.0
4  338155803  2022-01-18T00:00:04+00:00    0.0        99.0
5  366857070  2022-01-23T15:00:00+00:00    NaN        60.0
6  366857070  2022-01-19T19:02:56+00:00    NaN        60.0
7  366857070  2022-01-18T19:00:02+00:00    NaN        60.0
8  366857070  2022-01-14T03:00:04+00:00    NaN        60.0
9  338155803  2022-01-20T03:00:02+00:00    0.0        99.0

Data types:
MMSI                 int64
TrackStartTime      object
Draft              float64
VesselType         float64
geometry          geometry
dtype: object


In [ ]:
#---------------------------------------------------------------------------------------------
# 2023 data load
#---------------------------------------------------------------------------------------------


import geopandas as gpd
import pandas as pd
import time
import os
import fiona
from shapely.geometry import shape

# --- Configuration for 2023 ---
PATH_2023 = '/content/drive/MyDrive/SupplyChainPairINT/AIS_Data/extracted/AISVesselTracks2023.gpkg'
print("Exists:", os.path.exists(PATH_2023))
OUTPUT_PARQUET_DIR_2023 = '/content/drive/MyDrive/SupplyChainPairINT/AIS_Data/processed'
CHUNK_SIZE = 1000000
COLUMNS_TO_LOAD = ['MMSI', 'TrackStartTime', 'Draft', 'VesselType', 'geometry']
BBOX_LA_LB = (-119.0, 33.0, -117.0, 34.0)

os.makedirs(OUTPUT_PARQUET_DIR_2023, exist_ok=True)
print(f"Output directory created: {OUTPUT_PARQUET_DIR_2023}")

# --- Chunked Reading, Filtering, and Saving for 2023 ---
print(f"Starting chunked processing for 2023...")
start_total = time.time()
chunk_index = 0
total_features_processed = 0

try:
    with fiona.open(PATH_2023, 'r', layer=0) as source:

        output_crs = source.crs
        current_data = []

        for feature in source.filter(bbox=BBOX_LA_LB):
            properties = {k: feature['properties'][k] for k in COLUMNS_TO_LOAD if k != 'geometry'}

            current_data.append({
                'geometry': shape(feature['geometry']),
                **properties
            })

            total_features_processed += 1

            if len(current_data) >= CHUNK_SIZE:
                chunk_index += 1
                start_chunk = time.time()

                gdf_chunk = gpd.GeoDataFrame(current_data, geometry='geometry', crs=output_crs)
                output_file = os.path.join(OUTPUT_PARQUET_DIR_2023, f'2023_chunk_{chunk_index:03d}.parquet')
                gdf_chunk.to_parquet(output_file, index=False)

                print(f"   Saved Chunk {chunk_index} ({len(gdf_chunk):,} features) in {time.time() - start_chunk:.2f}s")
                current_data = []

    if current_data:
        chunk_index += 1
        start_chunk = time.time()
        gdf_chunk = gpd.GeoDataFrame(current_data, geometry='geometry', crs=output_crs)
        output_file = os.path.join(OUTPUT_PARQUET_DIR_2023, f'2023_chunk_{chunk_index:03d}.parquet')
        gdf_chunk.to_parquet(output_file, index=False)
        print(f"   Saved Final Chunk {chunk_index} ({len(gdf_chunk):,} features) in {time.time() - start_chunk:.2f}s")

    total_time = time.time() - start_total
    print(f"\n--- 2023 Processing Complete ---")
    print(f"Total time taken: {total_time:.2f} seconds")
    print(f"Total features filtered and processed: {total_features_processed:,}")

except Exception as e:
    print(f"An error occurred during 2023 processing: {e}")

Exists: True
Output directory created: /content/drive/MyDrive/SupplyChainPairINT/AIS_Data/processed
Starting chunked processing for 2023...


   Saved Final Chunk 1 (161,900 features) in 7.48s

--- 2023 Processing Complete ---
Total time taken: 103.49 seconds
Total features filtered and processed: 161,900


In [ ]:
#---------------------------------------------------------------------------------------------
# 2023 data view
#---------------------------------------------------------------------------------------------

import geopandas as gpd
import pandas as pd
import os

# --- Configuration ---
# Path to the directory where your 2021 data was saved
PARQUET_DIR = '/content/drive/MyDrive/SupplyChainPairINT/AIS_Data/processed'

# The filename will be '2023_chunk_001.parquet'
parquet_file = os.path.join(PARQUET_DIR, '2023_chunk_001.parquet')

# FIX: Add 'geometry' to the list of columns to satisfy GeoPandas
COLUMNS_TO_VIEW = ['MMSI', 'TrackStartTime', 'Draft', 'VesselType', 'geometry']

if os.path.exists(parquet_file):
    print(f"Loading sample from: {parquet_file}")

    # Use geopandas to read the data, including the geometry column
    sample_data = gpd.read_parquet(
        parquet_file,
        columns=COLUMNS_TO_VIEW
    ).head(10) # Read only the first 10 rows

    print("\n--- Sample Data from 2023 LA/LB Parquet (First 10 Rows) ---")

    # Display the data. We'll show the DataFrame without the complex geometry for clarity,
    # but it confirms the column exists.
    print(sample_data.drop(columns=['geometry']).to_string())

    print(f"\nData types:\n{sample_data.dtypes}")
else:
    print(f"Error: Parquet file not found at {parquet_file}. Check the folder contents.")

Loading sample from: /content/drive/MyDrive/SupplyChainPairINT/AIS_Data/processed/2023_chunk_001.parquet

--- Sample Data from 2023 LA/LB Parquet (First 10 Rows) ---
        MMSI             TrackStartTime  Draft  VesselType
0  338183593  2023-08-08T19:18:45+00:00    NaN        36.0
1  338179758  2023-08-06T20:59:08+00:00    NaN        37.0
2  338318302  2023-08-11T15:08:14+00:00    NaN        36.0
3  367647580  2023-08-09T18:07:49+00:00    NaN        37.0
4  366855060  2023-08-12T23:42:44+00:00    NaN        60.0
5  338318302  2023-08-13T20:53:55+00:00    NaN        36.0
6  338179758  2023-08-06T20:53:38+00:00    NaN        37.0
7  338179758  2023-08-03T20:27:27+00:00    NaN        37.0
8  338179758  2023-08-31T01:36:30+00:00    NaN        37.0
9  338213838  2023-08-20T14:26:30+00:00    NaN        37.0

Data types:
MMSI                 int64
TrackStartTime      object
Draft              float64
VesselType         float64
geometry          geometry
dtype: object


In [ ]:
#---------------------------------------------------------------------------------------------
# FINAL AIS DATA COMBINATION & CLEANING (2020–2022)
#---------------------------------------------------------------------------------------------

import geopandas as gpd
import pandas as pd
import os
import glob
import time

# --- Configuration ---
BASE_DIR = '/content/drive/MyDrive/SupplyChainPairINT/AIS_Data/processed'
OUTPUT_FILE = os.path.join(BASE_DIR, 'LA_LB_AIS_2021_2023_CLEAN_FINAL.parquet')

# File paths for each year (updated for your finalized file structure)
DATA_PATHS = {
    '2021': os.path.join(BASE_DIR, '2021_chunk_001.parquet'),
    '2022': os.path.join(BASE_DIR, '2022_chunk_001.parquet'),
    '2023': os.path.join(BASE_DIR, '2023_chunk_001.parquet'),
}

# Define commercial vessel type code range (Cargo & Tanker)
COMMERCIAL_VESSEL_MIN = 70.0
COMMERCIAL_VESSEL_MAX = 89.9

#---------------------------------------------------------------------------------------------
# Functions
#---------------------------------------------------------------------------------------------

def load_and_standardize_data(year):
    """
    Loads AIS data for a given year, fixes inconsistent column names, and standardizes formats.
    """
    path = DATA_PATHS[year]

    # Use glob to handle single or multiple parquet chunks
    file_list = glob.glob(path)

    if not file_list:
        if os.path.isfile(path):
            file_list = [path]
        else:
            print(f"❌ No files found for {year} at {path}")
            return gpd.GeoDataFrame()

    print(f"\n📦 Loading {len(file_list)} file(s) for {year}...")

    # Load all parquet files for the year
    gdfs = [gpd.read_parquet(f) for f in file_list]
    df = pd.concat(gdfs, ignore_index=True)

    # -----------------------------------------------------------------------------------------
    # Column Standardization & Time Column Fixes
    # -----------------------------------------------------------------------------------------
    if 'TrackStartTime' not in df.columns:
        print(f"⚠️ {year}: Missing 'TrackStartTime'. Attempting to locate correct time column...")
        for alt_col in ['DateTime', 'BaseDateTime', 'Time', 'time']:
            if alt_col in df.columns:
                print(f"   → Found '{alt_col}', renaming to 'TrackStartTime'.")
                df.rename(columns={alt_col: 'TrackStartTime'}, inplace=True)
                break
        else:
            print(f"   ❌ No recognizable time column found for {year}. Skipping this dataset.")
            return gpd.GeoDataFrame()

    # -----------------------------------------------------------------------------------------
    # Standard type conversions
    # -----------------------------------------------------------------------------------------
    df['TrackStartTime'] = pd.to_datetime(df['TrackStartTime'], utc=True, errors='coerce')

    # MMSI should be integer (some Parquet versions save as float)
    if 'MMSI' in df.columns:
        df['MMSI'] = pd.to_numeric(df['MMSI'], errors='coerce').astype('Int32')

    # Ensure VesselType and Draft exist and are numeric
    for col in ['VesselType', 'Draft']:
        if col not in df.columns:
            print(f"⚠️ {year}: Column '{col}' missing — filling with NaN.")
            df[col] = pd.NA
        else:
            df[col] = pd.to_numeric(df[col], errors='coerce')

    print(f"✅ Loaded {year}: {len(df):,} records.")
    return df


def perform_final_cleaning(df):
    """
    Applies filtering and final transformations to combined AIS data.
    """
    original_count = len(df)
    print(f"\n🧹 Starting cleaning on {original_count:,} total records...")

    # 1️⃣ Filter by commercial vessel type range
    df_cleaned = df[
        (df['VesselType'] >= COMMERCIAL_VESSEL_MIN) &
        (df['VesselType'] <= COMMERCIAL_VESSEL_MAX)
    ].copy()

    print(f"   → Retained {len(df_cleaned):,} after VesselType filter (70–89).")

    # 2️⃣ Draft cleaning and conversion
    df_cleaned.loc[df_cleaned['Draft'] <= 0.0, 'Draft'] = pd.NA
    df_cleaned['Draft_Meters'] = df_cleaned['Draft'] / 10.0
    df_cleaned.drop(columns=['Draft'], inplace=True)

    # 3️⃣ Drop missing key fields
    df_cleaned.dropna(subset=['MMSI', 'TrackStartTime', 'VesselType'], inplace=True)

    final_count = len(df_cleaned)
    print(f"   → Final cleaned record count: {final_count:,}")
    print(f"   → Total records removed: {original_count - final_count:,}")

    return df_cleaned


#---------------------------------------------------------------------------------------------
# Main Execution
#---------------------------------------------------------------------------------------------
if __name__ == '__main__':
    start_total = time.time()

    # Load and combine all years
    all_data = [load_and_standardize_data(year) for year in [ '2021', '2022', '2023']]
    all_data = [df for df in all_data if not df.empty]

    if not all_data:
        print("\n❌ FATAL: No valid data loaded. Check your file paths and formats.")
    else:
        combined_gdf = pd.concat(all_data, ignore_index=True)
        print(f"\n📊 Combined dataset size: {len(combined_gdf):,} records.")

        # Clean and filter
        cleaned_gdf = perform_final_cleaning(combined_gdf)

        # Save to final Parquet
        print(f"\n💾 Saving cleaned dataset to: {OUTPUT_FILE}")
        cleaned_gdf.to_parquet(OUTPUT_FILE, index=False)

        total_time = time.time() - start_total
        print("\n=======================================================")
        print("✅ FINAL CLEANED DATASET READY")
        print(f"Total processing time: {total_time:.2f} seconds")
        print(f"Output file: {OUTPUT_FILE}")
        print("=======================================================")

        # Show preview (without geometry)
        sample = cleaned_gdf.head().drop(columns=['geometry'], errors='ignore')
        print("\n🧾 Sample of final columns and data:")
        print(sample.to_string(index=False))
        print("\n🧱 Data types:")
        print(cleaned_gdf.dtypes)



📦 Loading 1 file(s) for 2021...
✅ Loaded 2021: 242,943 records.

📦 Loading 1 file(s) for 2022...
✅ Loaded 2022: 269,514 records.

📦 Loading 1 file(s) for 2023...
✅ Loaded 2023: 161,900 records.

📊 Combined dataset size: 674,357 records.

🧹 Starting cleaning on 674,357 total records...
   → Retained 143,057 after VesselType filter (70–89).
   → Final cleaned record count: 143,057
   → Total records removed: 531,300

💾 Saving cleaned dataset to: /content/drive/MyDrive/SupplyChainPairINT/AIS_Data/processed/LA_LB_AIS_2021_2023_CLEAN_FINAL.parquet

✅ FINAL CLEANED DATASET READY
Total processing time: 30.92 seconds
Output file: /content/drive/MyDrive/SupplyChainPairINT/AIS_Data/processed/LA_LB_AIS_2021_2023_CLEAN_FINAL.parquet

🧾 Sample of final columns and data:
     MMSI            TrackStartTime  VesselType  Draft_Meters
636016093 2021-03-19 14:54:59+00:00        80.0          1.56
357051000 2021-01-17 05:44:57+00:00        70.0          1.60
636017753 2021-01-17 06:20:14+00:00        70

In [ ]:
import geopandas as gpd

# Path to your final file
parquet_path = '/content/drive/MyDrive/SupplyChainPairINT/processed/LA_LB_AIS_2020_2022_CLEAN_FINAL.parquet'

# Load the Parquet file as a GeoDataFrame
gdf = gpd.read_parquet(parquet_path)

# Display first 5 rows (excluding geometry for readability)
print(gdf.head().drop(columns=['geometry'], errors='ignore'))

# If you want to verify it's a GeoDataFrame and see info:
print("\nDataFrame Info:")
print(gdf.info())




---


THis is the splitting point beyond this is the past dataset thingy



---
qwertyuiopppppppplkjhgfdsazxcvbnm,qwertyuiopasdfghjklzxcvbnm,qwertyuiopasdfghjkxcvbnm,qwertyuiopppppppplkjhgfdsazxcvbnm,qwertyuiopasdfghjklzxcvbnm,qwertyuiopasdfghjkxcvbnm,qwertyuiopppppppplkjhgfdsazxcvbnm,qwertyuiopasdfghjklzxcvbnm,qwertyuiopasdfghjkxcvbnm,qwertyuiopppppppplkjhgfdsazxcvbnm,qwertyuiopasdfghjklzxcvbnm,qwertyuiopasdfghjkxcvbnm,qwertyuiopppppppplkjhgfdsazxcvbnm,qwertyuiopasdfghjklzxcvbnm,qwertyuiopasdfghjkxcvbnm,


In [ ]:
#---------------------------------------------------------------------------------------------
# 2020 load
#---------------------------------------------------------------------------------------------


import geopandas as gpd
import fiona
import time
import os
import pandas as pd

# --- Configuration for 2020 GDB ---
PATH_2020_GDB = '/content/drive/MyDrive/SupplyChainPairINT/extracted/AISVesselTracks2020.gdb'
# We will overwrite the old file with one containing ALL columns
OUTPUT_PARQUET_2020_TEMP = '/content/drive/MyDrive/SupplyChainPairINT/processed/2020_LA_LB_TEMP_ALL_COLS.parquet'

# The LA/LB Bounding Box
BBOX_LA_LB = (-119.0, 33.0, -117.0, 34.0)

print(f"Starting RE-LOAD for 2020 GDB file, loading ALL columns for inspection...")
start_total = time.time()

try:
    # 1. Find the layer name (GDBs require this)
    layer_name = fiona.listlayers(PATH_2020_GDB)[0]

    # 2. Load all columns with BBOX filtering
    # NOTE: We skip the 'columns' parameter to load every attribute
    data_2020 = gpd.read_file(
        PATH_2020_GDB,
        layer=layer_name,
        bbox=BBOX_LA_LB
    )

    # 3. Save directly to a single TEMPORARY Parquet file
    data_2020.to_parquet(OUTPUT_PARQUET_2020_TEMP, index=False)

    total_time = time.time() - start_total
    print(f"✅ 2020 GDB TEMPORARY file created successfully with ALL columns.")
    print(f"Total records: {len(data_2020):,}")
    print(f"Total time taken: {total_time:.2f} seconds")

    # 4. Print all column names now that they are loaded!
    print("\n--- IDENTIFYING THE TIME COLUMN ---")
    print("ALL COLUMN NAMES in 2020 Data:")

    # Identify the time column by looking for time/date related names
    time_cols = [col for col in data_2020.columns if 'time' in col.lower() or 'date' in col.lower()]

    if time_cols:
        print(f"POSSIBLE TIME COLUMNS FOUND: {time_cols}")
        print(f"We will assume the first one is correct: {time_cols[0]}")
    else:
        print("No time or date columns found based on keywords. Printing entire list:")

    print(data_2020.columns.tolist())

except Exception as e:
    print(f"A final error occurred during 2020 GDB processing: {e}")

Starting RE-LOAD for 2020 GDB file, loading ALL columns for inspection...
✅ 2020 GDB TEMPORARY file created successfully with ALL columns.
Total records: 193,092
Total time taken: 20.32 seconds

--- IDENTIFYING THE TIME COLUMN ---
ALL COLUMN NAMES in 2020 Data:
POSSIBLE TIME COLUMNS FOUND: ['TrackStartTime', 'TrackEndTime']
We will assume the first one is correct: TrackStartTime
['MMSI', 'TrackStartTime', 'TrackEndTime', 'VesselType', 'Length', 'Width', 'Draft', 'DurationMinutes', 'VesselGroup', 'Shape_Length', 'geometry']


In [ ]:
#---------------------------------------------------------------------------------------------
# 2020 CLEAN LOAD (robust + version-compatible)
#---------------------------------------------------------------------------------------------

import geopandas as gpd
import fiona
import time
import os

# --- Configuration for 2020 GDB ---
PATH_2020_GDB = '/content/drive/MyDrive/SupplyChainPairINT/extracted/AISVesselTracks2020.gdb'

# Final clean Parquet file
OUTPUT_PARQUET_2020_FINAL = '/content/drive/MyDrive/SupplyChainPairINT/processed/2020_LA_LB_FINAL.parquet'

# Columns we actually need
CORRECT_TIME_COL = 'TrackStartTime'
COLUMNS_TO_LOAD_2020 = ['MMSI', CORRECT_TIME_COL, 'Draft', 'VesselType', 'geometry']

# LA/LB Bounding Box
BBOX_LA_LB = (-119.0, 33.0, -117.0, 34.0)

print("🚀 Starting FINAL CLEAN LOAD for 2020 GDB file (robust version)...")
start_total = time.time()

try:
    # 1️⃣ Identify layer name from GDB
    layer_name = fiona.listlayers(PATH_2020_GDB)[0]

    # 2️⃣ Try fast load with 'columns' parameter
    try:
        print("Attempting fast selective column load...")
        data_2020 = gpd.read_file(
            PATH_2020_GDB,
            layer=layer_name,
            columns=COLUMNS_TO_LOAD_2020,
            bbox=BBOX_LA_LB
        )
    except TypeError:
        # Fallback for GeoPandas versions without 'columns' support
        print("⚠️ 'columns' not supported — falling back to full load + manual subset.")
        data_2020 = gpd.read_file(
            PATH_2020_GDB,
            layer=layer_name,
            bbox=BBOX_LA_LB
        )
        # Keep only the needed columns
        available_cols = [c for c in COLUMNS_TO_LOAD_2020 if c in data_2020.columns]
        missing_cols = set(COLUMNS_TO_LOAD_2020) - set(available_cols)
        if missing_cols:
            print(f"⚠️ Warning: Missing columns in source: {missing_cols}")
        data_2020 = data_2020[available_cols]

    # 3️⃣ Save to final Parquet
    data_2020.to_parquet(OUTPUT_PARQUET_2020_FINAL, index=False)

    total_time = time.time() - start_total
    print("\n✅ 2020 DATA PREPARATION COMPLETE.")
    print(f"Records processed: {len(data_2020):,}")
    print(f"File saved to: {OUTPUT_PARQUET_2020_FINAL}")
    print(f"Time taken: {total_time:.2f} seconds")

except Exception as e:
    print(f"❌ Error during 2020 GDB processing: {e}")


🚀 Starting FINAL CLEAN LOAD for 2020 GDB file (robust version)...
Attempting fast selective column load...

✅ 2020 DATA PREPARATION COMPLETE.
Records processed: 193,092
File saved to: /content/drive/MyDrive/SupplyChainPairINT/processed/2020_LA_LB_FINAL.parquet
Time taken: 17.73 seconds


In [ ]:
#---------------------------------------------------------------------------------------------
# combine?
#---------------------------------------------------------------------------------------------


import geopandas as gpd
import os

# --- Configuration ---
PARQUET_DIR = '/content/drive/MyDrive/SupplyChainPairINT/processed'
parquet_file = os.path.join(PARQUET_DIR, '2020_LA_LB_FINAL.parquet')

# The final, correct list of columns
COLUMNS_TO_VIEW = ['MMSI', 'TrackStartTime', 'Draft', 'VesselType', 'geometry']

print(f"Loading sample from: {parquet_file}")

if os.path.exists(parquet_file):
    try:
        # Read the file, selecting the final set of columns
        sample_data = gpd.read_parquet(
            parquet_file,
            columns=COLUMNS_TO_VIEW
        ).head(10) # Read only the first 10 rows

        print("\n--- Final 2020 Data Sample (First 10 Rows) ---")

        # Display the data without the complex geometry column for clean output
        print(sample_data.drop(columns=['geometry']).to_string())

        print(f"\nData types:\n{sample_data.dtypes}")

    except Exception as e:
        print(f"Error reading the final file (Column mismatch likely): {e}")

else:
    print(f"Error: Parquet file not found at {parquet_file}. Check the folder contents.")


Loading sample from: /content/drive/MyDrive/SupplyChainPairINT/processed/2020_LA_LB_FINAL.parquet

--- Final 2020 Data Sample (First 10 Rows) ---
        MMSI            TrackStartTime  Draft  VesselType
0  210261000 2020-01-14 00:44:06+00:00   10.4        70.0
1  210261000 2020-01-14 09:59:44+00:00   10.4        70.0
2  211331640 2020-01-13 23:49:57+00:00   13.6        70.0
3  211331640 2020-01-14 00:12:27+00:00   13.6        70.0
4  211331640 2020-01-25 23:52:07+00:00   13.6        70.0
5  212350000 2020-01-31 04:34:36+00:00   14.0        70.0
6  212350000 2020-01-31 07:58:48+00:00   14.0        70.0
7  215560000 2020-01-01 00:02:05+00:00    NaN         NaN
8  218042000 2020-01-10 11:54:08+00:00   14.6        70.0
9  218042000 2020-01-13 14:01:10+00:00   14.6        70.0

Data types:
MMSI                            int32
TrackStartTime    datetime64[ms, UTC]
Draft                         float64
VesselType                    float64
geometry                     geometry
dtype: object

In [ ]:
#---------------------------------------------------------------------------------------------
# FINAL AIS DATA COMBINATION & CLEANING (2020–2022)
#---------------------------------------------------------------------------------------------

import geopandas as gpd
import pandas as pd
import os
import glob
import time

# --- Configuration ---
BASE_DIR = '/content/drive/MyDrive/SupplyChainPairINT/processed'
OUTPUT_FILE = os.path.join(BASE_DIR, 'LA_LB_AIS_2020_2022_CLEAN_FINAL.parquet')

# File paths for each year (updated for your finalized file structure)
DATA_PATHS = {
    '2020': os.path.join(BASE_DIR, '2020_LA_LB_FINAL.parquet'),
    '2021': os.path.join(BASE_DIR, '2021_chunk_001.parquet'),
    '2022': os.path.join(BASE_DIR, '2022_chunk_001.parquet'),
}

# Define commercial vessel type code range (Cargo & Tanker)
COMMERCIAL_VESSEL_MIN = 70.0
COMMERCIAL_VESSEL_MAX = 89.9

#---------------------------------------------------------------------------------------------
# Functions
#---------------------------------------------------------------------------------------------

def load_and_standardize_data(year):
    """
    Loads AIS data for a given year, fixes inconsistent column names, and standardizes formats.
    """
    path = DATA_PATHS[year]

    # Use glob to handle single or multiple parquet chunks
    file_list = glob.glob(path)

    if not file_list:
        if os.path.isfile(path):
            file_list = [path]
        else:
            print(f"❌ No files found for {year} at {path}")
            return gpd.GeoDataFrame()

    print(f"\n📦 Loading {len(file_list)} file(s) for {year}...")

    # Load all parquet files for the year
    gdfs = [gpd.read_parquet(f) for f in file_list]
    df = pd.concat(gdfs, ignore_index=True)

    # -----------------------------------------------------------------------------------------
    # Column Standardization & Time Column Fixes
    # -----------------------------------------------------------------------------------------
    if 'TrackStartTime' not in df.columns:
        print(f"⚠️ {year}: Missing 'TrackStartTime'. Attempting to locate correct time column...")
        for alt_col in ['DateTime', 'BaseDateTime', 'Time', 'time']:
            if alt_col in df.columns:
                print(f"   → Found '{alt_col}', renaming to 'TrackStartTime'.")
                df.rename(columns={alt_col: 'TrackStartTime'}, inplace=True)
                break
        else:
            print(f"   ❌ No recognizable time column found for {year}. Skipping this dataset.")
            return gpd.GeoDataFrame()

    # -----------------------------------------------------------------------------------------
    # Standard type conversions
    # -----------------------------------------------------------------------------------------
    df['TrackStartTime'] = pd.to_datetime(df['TrackStartTime'], utc=True, errors='coerce')

    # MMSI should be integer (some Parquet versions save as float)
    if 'MMSI' in df.columns:
        df['MMSI'] = pd.to_numeric(df['MMSI'], errors='coerce').astype('Int32')

    # Ensure VesselType and Draft exist and are numeric
    for col in ['VesselType', 'Draft']:
        if col not in df.columns:
            print(f"⚠️ {year}: Column '{col}' missing — filling with NaN.")
            df[col] = pd.NA
        else:
            df[col] = pd.to_numeric(df[col], errors='coerce')

    print(f"✅ Loaded {year}: {len(df):,} records.")
    return df


def perform_final_cleaning(df):
    """
    Applies filtering and final transformations to combined AIS data.
    """
    original_count = len(df)
    print(f"\n🧹 Starting cleaning on {original_count:,} total records...")

    # 1️⃣ Filter by commercial vessel type range
    df_cleaned = df[
        (df['VesselType'] >= COMMERCIAL_VESSEL_MIN) &
        (df['VesselType'] <= COMMERCIAL_VESSEL_MAX)
    ].copy()

    print(f"   → Retained {len(df_cleaned):,} after VesselType filter (70–89).")

    # 2️⃣ Draft cleaning and conversion
    df_cleaned.loc[df_cleaned['Draft'] <= 0.0, 'Draft'] = pd.NA
    df_cleaned['Draft_Meters'] = df_cleaned['Draft'] / 10.0
    df_cleaned.drop(columns=['Draft'], inplace=True)

    # 3️⃣ Drop missing key fields
    df_cleaned.dropna(subset=['MMSI', 'TrackStartTime', 'VesselType'], inplace=True)

    final_count = len(df_cleaned)
    print(f"   → Final cleaned record count: {final_count:,}")
    print(f"   → Total records removed: {original_count - final_count:,}")

    return df_cleaned


#---------------------------------------------------------------------------------------------
# Main Execution
#---------------------------------------------------------------------------------------------
if __name__ == '__main__':
    start_total = time.time()

    # Load and combine all years
    all_data = [load_and_standardize_data(year) for year in ['2020', '2021', '2022']]
    all_data = [df for df in all_data if not df.empty]

    if not all_data:
        print("\n❌ FATAL: No valid data loaded. Check your file paths and formats.")
    else:
        combined_gdf = pd.concat(all_data, ignore_index=True)
        print(f"\n📊 Combined dataset size: {len(combined_gdf):,} records.")

        # Clean and filter
        cleaned_gdf = perform_final_cleaning(combined_gdf)

        # Save to final Parquet
        print(f"\n💾 Saving cleaned dataset to: {OUTPUT_FILE}")
        cleaned_gdf.to_parquet(OUTPUT_FILE, index=False)

        total_time = time.time() - start_total
        print("\n=======================================================")
        print("✅ FINAL CLEANED DATASET READY")
        print(f"Total processing time: {total_time:.2f} seconds")
        print(f"Output file: {OUTPUT_FILE}")
        print("=======================================================")

        # Show preview (without geometry)
        sample = cleaned_gdf.head().drop(columns=['geometry'], errors='ignore')
        print("\n🧾 Sample of final columns and data:")
        print(sample.to_string(index=False))
        print("\n🧱 Data types:")
        print(cleaned_gdf.dtypes)



📦 Loading 1 file(s) for 2020...
✅ Loaded 2020: 193,092 records.

📦 Loading 1 file(s) for 2021...
✅ Loaded 2021: 242,943 records.

📦 Loading 1 file(s) for 2022...
✅ Loaded 2022: 269,514 records.

📊 Combined dataset size: 705,549 records.

🧹 Starting cleaning on 705,549 total records...
   → Retained 139,973 after VesselType filter (70–89).
   → Final cleaned record count: 139,973
   → Total records removed: 565,576

💾 Saving cleaned dataset to: /content/drive/MyDrive/SupplyChainPairINT/processed/LA_LB_AIS_2020_2022_CLEAN_FINAL.parquet

✅ FINAL CLEANED DATASET READY
Total processing time: 42.31 seconds
Output file: /content/drive/MyDrive/SupplyChainPairINT/processed/LA_LB_AIS_2020_2022_CLEAN_FINAL.parquet

🧾 Sample of final columns and data:
     MMSI            TrackStartTime  VesselType  Draft_Meters
210261000 2020-01-14 00:44:06+00:00        70.0          1.04
210261000 2020-01-14 09:59:44+00:00        70.0          1.04
211331640 2020-01-13 23:49:57+00:00        70.0          1.36
2

In [ ]:
import geopandas as gpd

# Path to your final file
parquet_path = '/content/drive/MyDrive/SupplyChainPairINT/processed/LA_LB_AIS_2020_2022_CLEAN_FINAL.parquet'

# Load the Parquet file as a GeoDataFrame
gdf = gpd.read_parquet(parquet_path)

# Display first 5 rows (excluding geometry for readability)
print(gdf.head().drop(columns=['geometry'], errors='ignore'))

# If you want to verify it's a GeoDataFrame and see info:
print("\nDataFrame Info:")
print(gdf.info())


        MMSI            TrackStartTime  VesselType  Draft_Meters
0  210261000 2020-01-14 00:44:06+00:00        70.0          1.04
1  210261000 2020-01-14 09:59:44+00:00        70.0          1.04
2  211331640 2020-01-13 23:49:57+00:00        70.0          1.36
3  211331640 2020-01-14 00:12:27+00:00        70.0          1.36
4  211331640 2020-01-25 23:52:07+00:00        70.0          1.36

DataFrame Info:
<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 139973 entries, 0 to 139972
Data columns (total 5 columns):
 #   Column          Non-Null Count   Dtype              
---  ------          --------------   -----              
 0   MMSI            139973 non-null  Int32              
 1   TrackStartTime  139973 non-null  datetime64[ns, UTC]
 2   VesselType      139973 non-null  float64            
 3   geometry        139973 non-null  geometry           
 4   Draft_Meters    127796 non-null  float64            
dtypes: Int32(1), datetime64[ns, UTC](1), float64(2), geometry(1)
mem